# 🧠 LeetCode 239: Sliding Window Maximum

**Pattern:** Monotonic Deque (Sliding Window)

- **Created:** 2026-02-28
- **Focus:** Keeping track of decreasing potential maximums over time
- **Tags:** `array` | `sliding-window` | `deque` | `hard` | `citi-prep`

## 📖 Problem Statement

You are given an array of integers `nums`, there is a sliding window of size `k` which is moving from the very left of the array to the very right. You can only see the `k` numbers in the window. Each time the sliding window moves right by one position.

Return the max sliding window.

### Example 1:
- **Input:** `nums = [1,3,-1,-3,5,3,6,7]`, `k = 3`
- **Output:** `[3,3,5,5,6,7]`
- **Explanation:** 
  - `[1  3  -1] -3  5 ...` Max = `3`
  - ` 1 [3  -1  -3] 5 ...` Max = `3`
  - ` 1  3 [-1  -3  5]...` Max = `5`

## 🧠 Mental Model & Intuition

Finding the max in O(k) for every single window shift means comparing things we already compared. The brute force is O(N*K).

**The "Line at the Club" Analogy:**
Think of the numbers in the window standing in an orderly line queue. 
When a new, muscular bouncer approaches (`5`), he looks at the people standing in line in front of him (`-1`, `-3`). Because he is bigger than them, and he arrived *after* them (meaning his ticket lasts longer before falling out of the window), the smaller people are objectively useless. They will *never* be the maximum while he is there. So he kicks them out of line.

We maintain a **Monotonically Decreasing Deque**. It stores indices. The values represented by the deque are always strictly decreasing from left to right. The maximum element currently in the window is ALWAYS sitting comfortably at the front (`deque[0]`).

## 🐢 Brute Force Approach

Slice the array $N-k$ times. Run the `max()` scan across $K$ elements every time.

```python
def maxSlidingWindowBrute(nums, k):
    res = []
    for i in range(len(nums) - k + 1):
        res.append(max(nums[i:i+k]))
    return res
```
**Time:** $O(N \times K)$ (Fails LeetCode Hard Time Constraints) | **Space:** $O(N - K)$ for output.

In [ ]:
# Optimal Monotonic Deque Approach
import collections

def maxSlidingWindow(nums: list[int], k: int) -> list[int]:
    res = []
    # Deque stores INDICES, not the values themselves.
    q = collections.deque()
    
    for r in range(len(nums)):
        # Phase 1: Evict elements from the BACK that are smaller than the incoming guy
        # (They are useless because the new guy is bigger and arrived later)
        while q and nums[q[-1]] < nums[r]:
            q.pop()
            
        q.append(r)
        
        # Phase 2: Evict elements from the FRONT that have fallen out of the window's left edge
        # The left edge of the window is (r - k + 1)
        if q[0] < r - k + 1:
            q.popleft()
            
        # Phase 3: Add to results if our window is at least size k
        if r >= k - 1:
            res.append(nums[q[0]]) # The max is ALWAYS at the front of the queue
            
    return res

print("Sliding Max [1,3,-1,-3,5,3,6,7], k=3: ", maxSlidingWindow([1,3,-1,-3,5,3,6,7], 3))

## ⏱️ Complexity Analysis

- **Time Complexity:** $O(N)$. Even with the `while` loop, every index is added to the deque exactly once and `pop()`ped at most once. The amortized cost per element is $O(1)$. Total time is $2N$.
- **Space Complexity:** $O(K)$. The deque will hold at most `k` indices at any given point (specifically in the worst-case scenario where elements are strictly descending `[5, 4, 3, 2, 1]`, nobody kicks anyone out, leaving the deque size equal to $k$).

## 🎤 Interview Q&A

### Q1: Why store indices in the Deque instead of the actual array values?
**Answer:** We need indices to know when an element has successfully "aged out" of the sliding window boundary. If we stored the value `3`, and the window shifted past a `3` in the array, we wouldn't know if the `3` at the front of our deque was the old one that expired, or a newer identical one we just evaluated. Comparing `q[0] < left_boundary_index` unambiguously tells us if the element has expired.

---

### Q2: Why is a regular List bad for this? Why specific `collections.deque`?
**Answer:** A basic Python list uses $O(N)$ time to perform `pop(0)` from the front of the array because it must re-index every following element in contiguous memory. A Deque (Doubly Ended Queue) is implemented as a doubly-linked list in C under the hood, making `popleft()` an $O(1)$ operation. In an algorithmic interview spanning thousands of loops, $O(N)$ vs $O(1)$ on popping is pass/fail.

## 📚 Key Terminology

| Term | Definition | Example |
|---|---|---|
| **Monotonic Queue** | A data structure ensuring elements are strictly increasing or decreasing. | Enforcing `[6, 4, 2]` |
| **Deque** | Doubly Ended Queue. Fast pushes and pops from both sides. | `collections.deque()` |
| **Amortized $O(1)$** | An operation that is occasionally slow (while loop popping many items), but over $N$ times averages out to a constant cost. | Popping limits $N$ times total. |

## 💼 The Citi Narrative

**Context:** Tick Volume Peaks.

**Scenario:** In algorithmic trading, systems gauge market volatility not just by current volume, but by the MAXIMUM volume spike seen in the last 15 seconds. New tick volume arrives every millisecond. We had a rolling 15,000-tick window over a streaming sequence of millions of ticks.

**Impact:** The naive calculation of calling `max()` on an array of 15,000 integers, every single millisecond, choked the telemetry parser. Implementing a Monotonic Deque in Java provided a guaranteed O(1) amortized lookup to dynamically retrieve the peak historical volatility spike within the sliding 15-second bounds, allowing real-time trading signals to resume unimpeded.

In [ ]:
# EXERCISE: Trace the Deque
# Array: [3, 1, -1, 4]  (K=3)
# Step 1: add 3. Deque=[3]
# Step 2: add 1. Deque=[3, 1] (Valid, it's decreasing)
# Step 3: add -1. Deque=[3, 1, -1] (Valid). Max is 3 (Queue front). Result=[3]
# Step 4: add 4. 4 kicks out -1, kicks out 1, kicks out 3. Deque=[4]. Result=[3, 4].
print("The muscle kicks out all weaker earlier arrivals.")

## 🎯 Summary: Key Takeaways

### The Pattern
**Monotonic Deque** — Eliminating smaller historic values from a list since incoming larger values make them functionally useless.

### When to Use It
- ✅ Real-time rolling max/min filters (e.g. trading volume spikes).
- ✅ Bounded expiration queues prioritizing magnitude over arrival time.
- ❌ **Don't use when:** Needing to track the median (Requires Two Heaps instead).

### Complexity
| Approach | Time | Space |
|----------|------|-------|
| Brute Force | $O(N \times K)$ | $O(N-K)$ |
| Optimal | $O(N)$ | $O(K)$ |

### Interview Confidence Checklist
- [ ] Can explain the brute force and why it fails
- [ ] Can state the pattern name and core insight in one sentence
- [ ] Can write the optimal solution from memory
- [ ] Can state time and space complexity with justification
- [ ] Can name at least one real-world / Citi application

---

**"Simplicity and clarity is Gold."** — Sean's Study Mantra

Master **Monotonic Deque** and you've mastered one of the most common patterns in data engineering interviews. 🚀